In [1]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import matplotlib.ticker as ticker

# ---------- Единый стиль для всех графиков ----------
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)

TEXT_COLOR = "#000000"
GRID_COLOR = "#D9D9D9"
AVG_COLOR = "#000000"
DAU_COLOR = "#3B82F6"
HOURS_COLOR = "#F59E0B"
ORGANIC_COLOR = "#2E8B57"
EXTRA_COLOR = "#7C3AED"

SEGMENT_ORDER = ["Cold", "Passive", "Explorer", "Mixed"]
SEGMENT_COLORS = {
    "Cold": "#4C78A8",
    "Passive": "#F58518",
    "Explorer": "#54A24B",
    "Mixed": "#B279A2",
}

TIER_ORDER = ["Head", "Torso", "Tail"]
TIER_COLORS = {
    "Head": "#4C78A8",
    "Torso": "#F2CF5B",
    "Tail": "#E45756",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "text.color": TEXT_COLOR,
    "axes.labelcolor": TEXT_COLOR,
    "axes.titlecolor": TEXT_COLOR,
    "xtick.color": TEXT_COLOR,
    "ytick.color": TEXT_COLOR,
    "legend.labelcolor": TEXT_COLOR,
    "axes.edgecolor": "#BDBDBD",
})


def style_axis(ax):
    """Единое оформление осей: белый фон, черный текст, спокойная сетка."""
    ax.tick_params(axis="both", colors=TEXT_COLOR)
    ax.xaxis.label.set_color(TEXT_COLOR)
    ax.yaxis.label.set_color(TEXT_COLOR)
    ax.title.set_color(TEXT_COLOR)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="y", color=GRID_COLOR, alpha=0.7)
    ax.grid(axis="x", visible=False)


def add_caption(ax, text, y=-0.23):
    """Пояснение под графиком — всегда черным цветом."""
    ax.text(
        0, y, text,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=10.5,
        color=TEXT_COLOR,
        wrap=True,
    )


MARTS_DIR = Path.cwd() / "data" / "marts"

# Загружаем витрину
df_daily = pl.read_parquet(MARTS_DIR / "mart_daily_metrics.parquet")
pd_daily = df_daily.select(["time_period", "dau", "played_hours"]).to_pandas()

# ---------- Агрегация по неделям ----------
pd_daily["week"] = pd_daily["time_period"] // 7
pd_weekly = pd_daily.groupby("week").agg({
    "dau": "mean",
    "played_hours": "mean",
}).reset_index()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# 1. Аудитория
ax1.plot(pd_weekly["week"], pd_weekly["dau"], color=DAU_COLOR, linewidth=2.5)
ax1.fill_between(pd_weekly["week"], pd_weekly["dau"], alpha=0.12, color=DAU_COLOR)

mean_dau = pd_weekly["dau"].mean()
ax1.axhline(
    mean_dau,
    linestyle="--",
    linewidth=1.8,
    color=AVG_COLOR,
    label=f"Среднее за всё время: {mean_dau:,.0f}".replace(",", " "),
)
ax1.legend(loc="lower right", frameon=True, facecolor="white")
ax1.set_title("Аудитория: средний DAU по неделям", loc="left", fontsize=15, fontweight="bold")
ax1.set_ylabel("Активные пользователи в день")
ax1.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: f"{x / 1000:.1f}K" if x >= 1000 else f"{x:.0f}")
)
style_axis(ax1)
add_caption(
    ax1,
    "Показывает, как менялось среднее число активных пользователей за день. "
    "Пунктирная линия — среднее значение DAU за весь период.",
    y=-0.18,
)

# 2. Вовлеченность
ax2.plot(pd_weekly["week"], pd_weekly["played_hours"], color=HOURS_COLOR, linewidth=2.5)
ax2.fill_between(pd_weekly["week"], pd_weekly["played_hours"], alpha=0.12, color=HOURS_COLOR)

mean_hours = pd_weekly["played_hours"].mean()
ax2.axhline(
    mean_hours,
    color=AVG_COLOR,
    linestyle="--",
    linewidth=1.8,
    label=f"Среднее за всё время: {mean_hours:,.0f}".replace(",", " "),
)
ax2.legend(loc="lower right", frameon=True, facecolor="white")
ax2.set_title("Вовлеченность: среднедневные часы прослушивания", loc="left", fontsize=15, fontweight="bold")
ax2.set_xlabel("Неделя")
ax2.set_ylabel("Часы прослушивания в день")
ax2.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: f"{x / 1000:.1f}K" if x >= 1000 else f"{x:.0f}")
)
style_axis(ax2)
add_caption(
    ax2,
    "Показывает, как менялся общий объем прослушивания в день. "
    "Пунктирная линия — среднее значение за весь период.",
    y=-0.26,
)

fig.subplots_adjust(hspace=0.55, bottom=0.14)
plt.show()

FileNotFoundError: Системе не удается найти указанный путь. (os error 3): c:/Users/user/Downloads/data/marts/mart_daily_metrics.parquet

In [ ]:
# ---------- Доля органических прослушиваний ----------
pd_org = df_daily.select(["time_period", "organic_listen_ratio"]).to_pandas()
pd_org["week"] = pd_org["time_period"] // 7
pd_org_weekly = pd_org.groupby("week")["organic_listen_ratio"].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 5.5))

ax.plot(
    pd_org_weekly["week"],
    pd_org_weekly["organic_listen_ratio"],
    color=ORGANIC_COLOR,
    linewidth=2.5,
)
ax.fill_between(
    pd_org_weekly["week"],
    pd_org_weekly["organic_listen_ratio"],
    alpha=0.12,
    color=ORGANIC_COLOR,
)

mean_org = pd_org_weekly["organic_listen_ratio"].mean()
ax.axhline(
    mean_org,
    color=AVG_COLOR,
    linestyle="--",
    linewidth=1.8,
    label=f"Среднее за всё время: {mean_org:.1%}",
)

ax.set_title("Доля органических прослушиваний по неделям", loc="left", fontsize=15, fontweight="bold")
ax.set_xlabel("Неделя")
ax.set_ylabel("Доля органических прослушиваний")
ax.yaxis.set_major_formatter(ticker.PercentFormatter(1.0))
ax.legend(loc="lower right", frameon=True, facecolor="white")
style_axis(ax)
add_caption(
    ax,
    "Показывает, какая доля прослушиваний приходится на органическое потребление контента. "
    "Пунктирная линия — средняя доля за весь период.",
    y=-0.25,
)

fig.subplots_adjust(bottom=0.22)
plt.show()

In [ ]:
# ---------- Пользовательские сегменты ----------
pd_users = df_users.to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

# 1. Размер сегментов
sns.countplot(
    data=pd_users,
    x="segment",
    hue="segment",
    order=SEGMENT_ORDER,
    hue_order=SEGMENT_ORDER,
    palette=SEGMENT_COLORS,
    legend=False,
    ax=axes[0],
)
axes[0].set_title("Размер пользовательских сегментов", loc="left", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Количество пользователей")
axes[0].set_xlabel("Сегмент")
style_axis(axes[0])
add_caption(
    axes[0],
    "Показывает, сколько пользователей относится к каждому поведенческому сегменту.",
    y=-0.23,
)

# 2. Дослушиваемость по сегментам
sns.boxplot(
    data=pd_users,
    x="segment",
    y="completion_rate",
    hue="segment",
    order=SEGMENT_ORDER,
    hue_order=SEGMENT_ORDER,
    palette=SEGMENT_COLORS,
    legend=False,
    ax=axes[1],
)
axes[1].set_title("Дослушиваемость треков по сегментам", loc="left", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Доля дослушивания")
axes[1].set_xlabel("Сегмент")
axes[1].set_ylim(0, 1)
axes[1].yaxis.set_major_formatter(ticker.PercentFormatter(1.0))
style_axis(axes[1])
add_caption(
    axes[1],
    "Показывает распределение доли дослушивания в каждом сегменте. "
    "Линия внутри коробки — медиана, коробка — центральные 50% пользователей.",
    y=-0.23,
)

fig.subplots_adjust(wspace=0.22, bottom=0.20)
plt.show()

In [ ]:
# ---------- Популярность контента и скипы ----------
pd_items = df_items.to_pandas()

# Суммарные прослушивания по тирам в фиксированном порядке
tier_stats = (
    pd_items.groupby("content_tier", as_index=False)["listens"].sum()
    .set_index("content_tier")
    .reindex(TIER_ORDER)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

# 1. Доля прослушиваний
pie_colors = [TIER_COLORS[tier] for tier in TIER_ORDER]
axes[0].pie(
    tier_stats["listens"],
    labels=tier_stats["content_tier"],
    autopct="%1.1f%%",
    colors=pie_colors,
    startangle=90,
    textprops={"color": TEXT_COLOR, "fontsize": 11},
    wedgeprops={"edgecolor": "white", "linewidth": 1.2},
)
axes[0].set_title("Доля прослушиваний по типу контента", loc="left", fontsize=14, fontweight="bold", color=TEXT_COLOR)
add_caption(
    axes[0],
    "Показывает, какая часть всех прослушиваний приходится на Head, Torso и Tail-контент.",
    y=-0.13,
)

# 2. Доля быстрых скипов
sns.barplot(
    data=pd_items,
    x="content_tier",
    y="short_listen_rate",
    hue="content_tier",
    order=TIER_ORDER,
    hue_order=TIER_ORDER,
    palette=TIER_COLORS,
    errorbar=None,
    legend=False,
    ax=axes[1],
)
axes[1].set_title("Доля быстрых скипов по типу контента", loc="left", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Доля скипов (<10% трека)")
axes[1].set_xlabel("Тип контента")
axes[1].set_ylim(0, 1)
axes[1].yaxis.set_major_formatter(ticker.PercentFormatter(1.0))
style_axis(axes[1])
add_caption(
    axes[1],
    "Показывает среднюю долю прослушиваний, завершившихся до 10% трека, для каждого типа контента.",
    y=-0.23,
)

fig.subplots_adjust(wspace=0.24, bottom=0.20)
plt.show()

In [ ]:
# ---------- Дополнительный график: часы на одного активного пользователя ----------
# Этот показатель отделяет рост вовлеченности от простого роста аудитории.
pd_per_user = df_daily.select(["time_period", "dau", "played_hours"]).to_pandas()
pd_per_user = pd_per_user[pd_per_user["dau"] > 0].copy()
pd_per_user["week"] = pd_per_user["time_period"] // 7
pd_per_user["hours_per_user"] = pd_per_user["played_hours"] / pd_per_user["dau"]

pd_per_user_weekly = (
    pd_per_user.groupby("week")["hours_per_user"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.plot(
    pd_per_user_weekly["week"],
    pd_per_user_weekly["hours_per_user"],
    color=EXTRA_COLOR,
    linewidth=2.5,
)
ax.fill_between(
    pd_per_user_weekly["week"],
    pd_per_user_weekly["hours_per_user"],
    color=EXTRA_COLOR,
    alpha=0.12,
)

mean_hpu = pd_per_user_weekly["hours_per_user"].mean()
ax.axhline(
    mean_hpu,
    color=AVG_COLOR,
    linestyle="--",
    linewidth=1.8,
    label=f"Среднее за всё время: {mean_hpu:.2f} ч",
)

ax.set_title("Средние часы прослушивания на активного пользователя", loc="left", fontsize=15, fontweight="bold")
ax.set_xlabel("Неделя")
ax.set_ylabel("Часов на пользователя в день")
ax.legend(loc="lower right", frameon=True, facecolor="white")
style_axis(ax)
add_caption(
    ax,
    "Показывает вовлеченность без эффекта роста DAU. Если показатель растет, пользователи в среднем слушают больше; "
    "если он стабилен, рост общих часов в основном объясняется увеличением аудитории.",
    y=-0.25,
)

fig.subplots_adjust(bottom=0.22)
plt.show()